In [5]:
from pathlib import Path

for file in Path("Raw_Data_Malaysia").iterdir():
    print(file.name)


waqi-covid19-airqualitydata-2022Q1.csv
waqi-covid19-airqualitydata-2022Q2.csv
waqi-covid19-airqualitydata-2022Q3.csv
waqi-covid19-airqualitydata-2022Q4.csv
waqi-covid19-airqualitydata-2023Q1.csv
waqi-covid19-airqualitydata-2023Q2.csv
waqi-covid19-airqualitydata-2023Q3.csv
waqi-covid19-airqualitydata-2023Q4.csv
waqi-covid19-airqualitydata-2026.csv


## Combine And Clean Historical AQI Data

This step combines all CSV files from the `Raw_Data_Malaysia` folder into one dataset named `historical_AQI_data.csv`.

Cleaning actions performed:
- retain only Malaysia (`MY`) records prepared in `Raw_Data_Malaysia`
- remove metadata lines and repeated headers
- keep only the required AQI columns
- standardize text values by trimming spaces
- convert date and numeric columns into suitable data types
- remove incomplete rows and duplicate records
- sort the final dataset before saving for Hive upload


In [ ]:
from pathlib import Path
import pandas as pd

raw_data_folder = Path('Raw_Data_Malaysia')
output_file = Path('historical_AQI_data_malaysia_1.csv')
csv_files = sorted(raw_data_folder.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError('No CSV files were found in the Raw_Data_Malaysia folder')

dataframes = []

expected_columns = ['Date', 'Country', 'City', 'Specie', 'count', 'min', 'max', 'median', 'variance']

for file in csv_files:
    df = pd.read_csv(file, comment='#', dtype=str)
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all')
    df = df[expected_columns]
    dataframes.append(df)

historical_aqi_data = pd.concat(dataframes, ignore_index=True)

# Remove repeated header rows if they appear after combining files.
historical_aqi_data = historical_aqi_data[
    historical_aqi_data['Date'].astype(str).str.strip() != 'Date'
]

# Standardize text columns.
for column in ['Country', 'City', 'Specie']:
    historical_aqi_data[column] = historical_aqi_data[column].astype(str).str.strip()

# Convert columns to suitable data types.
historical_aqi_data['Date'] = pd.to_datetime(historical_aqi_data['Date'], errors='coerce')
for column in ['count', 'min', 'max', 'median', 'variance']:
    historical_aqi_data[column] = pd.to_numeric(historical_aqi_data[column], errors='coerce')

# Remove incomplete or duplicated records.
historical_aqi_data = historical_aqi_data.dropna(subset=['Date', 'Country', 'City', 'Specie'])
historical_aqi_data = historical_aqi_data.drop_duplicates()
historical_aqi_data = historical_aqi_data.sort_values(['Date', 'Country', 'City', 'Specie']).reset_index(drop=True)
historical_aqi_data['Date'] = historical_aqi_data['Date'].dt.strftime('%Y-%m-%d')

#historical_aqi_data.to_csv(output_file, index=False) #not required as we want to combine with even more data

print(f'Combined {len(csv_files)} files into {output_file.name}')
print('Files included:')
for file in csv_files:
    print(f'- {file.name}')
print(f'Total cleaned rows: {len(historical_aqi_data):,}')
historical_aqi_data.head()


Combined 9 files into historical_AQI_data_malaysia_1.csv
Files included:
- waqi-covid19-airqualitydata-2022Q1.csv
- waqi-covid19-airqualitydata-2022Q2.csv
- waqi-covid19-airqualitydata-2022Q3.csv
- waqi-covid19-airqualitydata-2022Q4.csv
- waqi-covid19-airqualitydata-2023Q1.csv
- waqi-covid19-airqualitydata-2023Q2.csv
- waqi-covid19-airqualitydata-2023Q3.csv
- waqi-covid19-airqualitydata-2023Q4.csv
- waqi-covid19-airqualitydata-2026.csv
Total cleaned rows: 114,218


,Date,Country,City,Specie,count,min,max,median,variance
0,2022-03-28,MY,Alor Setar,aqi,42,32.0,47.0,37.0,205.60
1,2022-03-28,MY,Alor Setar,dew,42,24.0,27.0,25.0,8.08
2,2022-03-28,MY,Alor Setar,humidity,48,66.0,100.0,88.0,1074.26
3,2022-03-28,MY,Alor Setar,pressure,48,1006.0,1012.0,1010.0,31.47
4,2022-03-28,MY,Alor Setar,temperature,48,25.0,33.0,27.0,78.87


In [7]:
historical_aqi_data.City.value_counts()

City
Malacca         9589
Kuala Lumpur    9357
Klang           9310
Alor Setar      8929
Kuantan         8746
Kota Bharu      8713
George Town     8623
Kuching         8554
Johor Bahru     8547
Seremban        8528
Taiping         8448
Ipoh            8441
Miri            8433
Name: count, dtype: int64

In [8]:
historical_aqi_data.Specie.value_counts()


Specie
humidity         17108
pressure         17108
temperature      17108
wind-speed       17108
dew              17102
aqi              15897
wind-gust         4698
pm10              1248
pm25              1248
no2               1244
so2               1244
co                1243
o3                1243
precipitation      619
Name: count, dtype: int64

In [12]:

# Need to run previous cell
base_data = historical_aqi_data.copy()


raw_data_folder_2 = Path('Raw_Data_Malaysia_2')
output_file = Path('Malaysia_AQI_data_combined.csv')
csv_files_2 = sorted(raw_data_folder_2.glob('*.csv'))

if not csv_files_2:
    raise FileNotFoundError('No CSV files were found in Raw_Data_Malaysia_2')

expected_base_columns = ['Date', 'Country', 'City', 'Specie', 'count', 'min', 'max', 'median', 'variance']
base_data.columns = base_data.columns.str.strip()
base_data = base_data[expected_base_columns].copy()

def city_from_filename(file_path: Path) -> str:
    name = file_path.stem.replace('-air-quality', '').strip()
    city = name.split(',')[0].strip(' -')
    return city.title()

extra_frames = []
for file in csv_files_2:
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip().str.lower()

    if 'date' not in df.columns:
        continue

    pollutant_columns = [c for c in df.columns if c not in ['date']]
    melted = df.melt(id_vars='date', value_vars=pollutant_columns, var_name='Specie', value_name='value')

    melted['Date'] = pd.to_datetime(melted['date'], errors='coerce')
    melted['value'] = pd.to_numeric(melted['value'], errors='coerce')
    melted = melted.dropna(subset=['Date', 'value'])

    if melted.empty:
        continue

    city_name = city_from_filename(file)
    transformed = pd.DataFrame({
        'Date': melted['Date'],
        'Country': 'Malaysia',
        'City': city_name,
        'Specie': melted['Specie'].astype(str).str.strip().str.lower(),
        'count': 1,
        'min': melted['value'],
        'max': melted['value'],
        'median': melted['value'],
        'variance': 0
    })
    extra_frames.append(transformed)

if not extra_frames:
    raise ValueError('Raw_Data_Malaysia_2 files were read, but no valid rows were produced.')

extra_data = pd.concat(extra_frames, ignore_index=True)
combined_data = pd.concat([base_data, extra_data], ignore_index=True)

for column in ['Country', 'City', 'Specie']:
    combined_data[column] = combined_data[column].astype(str).str.strip()

combined_data['Date'] = pd.to_datetime(combined_data['Date'], errors='coerce')
for column in ['count', 'min', 'max', 'median', 'variance']:
    combined_data[column] = pd.to_numeric(combined_data[column], errors='coerce')

combined_data = combined_data.dropna(subset=['Date', 'Country', 'City', 'Specie'])
combined_data = combined_data.drop_duplicates()
combined_data = combined_data.sort_values(['Date', 'Country', 'City', 'Specie']).reset_index(drop=True)
combined_data['Date'] = combined_data['Date'].dt.strftime('%Y-%m-%d')

combined_data.to_csv(output_file, index=False)

print(f'Added city files: {len(csv_files_2)}')
print(f'Rows from Raw_Data_Malaysia_2: {len(extra_data):,}')
print(f'Total rows in combined file: {len(combined_data):,}')
print(f'Output: {output_file.name}')
combined_data.head()


Added city files: 5
Rows from Raw_Data_Malaysia_2: 19,832
Total rows in combined file: 134,050
Output: Malaysia_AQI_data_combined.csv


,Date,Country,City,Specie,count,min,max,median,variance
0,2014-01-01,Malaysia,Banting,aqi,1,41.0,41.0,41.0,0.0
1,2014-01-01,Malaysia,Cheras,aqi,1,32.0,32.0,32.0,0.0
2,2014-01-01,Malaysia,Petaling-Jaya,aqi,1,40.0,40.0,40.0,0.0
3,2014-01-01,Malaysia,Shah-Alam,aqi,1,53.0,53.0,53.0,0.0
4,2014-01-02,Malaysia,Banting,aqi,1,39.0,39.0,39.0,0.0


In [13]:
combined_data.City.value_counts()

City
Malacca          9589
Kuala Lumpur     9357
Klang            9310
Alor Setar       8929
Kuantan          8746
Kota Bharu       8713
George Town      8623
Kuching          8554
Johor Bahru      8547
Seremban         8528
Taiping          8448
Ipoh             8441
Miri             8433
Cheras           4312
Banting          4253
Shah-Alam        4244
Petaling-Jaya    4238
Putrajaya        2785
Name: count, dtype: int64